### Choosing a model
---

Sentence-transformers supports many models. The right one depends on your task, your language, and the resources you have. Larger models are usually slower, so for our FAQ dataset of short English texts a small model is enough. Try a few on your own data and keep the one that works best.

We'll use all-MiniLM-L6-v2:

- 384-dimensional vectors (compact)
- Fast on CPU
- Good quality for general English text
- Uses cosine similarity (we'll explain this below)


In [2]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

The first time you run this, it downloads the model (~80 MB) and the tokenizer from HuggingFace. The tokenizer turns text into something the model can read. After that, both load from a local cache.

In [ ]:
q1 = "Can I still join the course after the start date?"
v1 = model.encode(q1)

v1.shape

# V1 is a vector with 384 numbers. Each number stands for some concept the model learned.

(384,)

In [7]:
# Encode our document:

d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [ ]:
# Next, we compare the query against the document using dot product:

v1.dot(dv) # Dot is vector multiplication

np.float32(0.32332402)

In [9]:
# Now we try an unrelated query:

q2 = "How to install Docker on Windows?"
v2 = model.encode(q2)

v2.dot(dv)

np.float32(0.01973048)

The first score for q1 vs d (0.32) is higher, so that query is more similar to the document about registration. The second score for q2 vs d sits near 0, because installing Docker has nothing to do with registration. A score near 0 means the two vectors are about as different as they can be.

That's the whole idea behind vector search: similar texts get similar vectors, and a dot product tells us how similar.

### Cosine similarity

The `all-MiniLM-L6-v2` model outputs normalized vectors - vectors with unit length. When both vectors are normalized, the dot product equals cosine similarity. That's why the model documentation says it "uses cosine similarity."

Cosine similarity measures the angle between two vectors, ignoring their length:

- 1.0 = same direction (similar)
- 0.0 = perpendicular (unrelated)
- -1.0 = opposite direction (opposite meaning)

formally, if `theta`is the angle between two vectors, cosine similarity is `cos(theta)`:

- `cos(0) = 1` - vectors point in the same direction
- `cos(90) = 0` - vectors are perpendicular
- `cos(180) = -1` - vectors point in opposite direction


Because our vectors are normalized, the dot product gives us cosine similarity directly. This is why we can use `v1.dot(dv)` to compare texts.

In practice, we rarely get cosine similarity below 0. The embedding model maps text to a region of the vector space where most vectors have positive components. There's no concept of "opposite meaning" that maps to a vector pointing the other way.

Now apply embeddings to the whole FAQ dataset

In [10]:
from ingest import load_faq_data
documents = load_faq_data()

### Generating embeddings
---

Each documents is a python dictionary with a question and an answer. We need to change it as text to make embeddings

In [21]:
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

len(texts)


1368

Now we generate the embeddings.

We have about 1300 texts here. We won't hand the model all of them at once. That takes a long time, and we can't see what's happening inside.

Instead we split them into batches

In [22]:
# First we import tqdm to see the progress
from tqdm import tqdm

# Next we chunk the texts into batches of 50 and encode each batch

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

# We end up with 1368 vectors.



100%|██████████| 28/28 [00:05<00:00,  5.37it/s]


1368

We turn them into a 2-dimensional array(matrix) where

- rows are documents (vectors)
- columns are dimensions of the vectors

In [24]:
import numpy as np
X = np.array(vectors)

X.shape

# Now we have 1368 vectors with 384 dimensions each.



(1368, 384)

We embedded our FAQ dataset into a matrix `X` with 1368 document vectors. Here's how vector search works under the hood

### Scoring documents
---

We have a matrix `X` with all document embeddings. We take a query, compare it against every document, and keep the most similar ones.


In [35]:
# When a query comes in, we embed it:

query = "Can I still join the course after the start date?"
v_query = model.encode(query)

# Next, we compute the dot product against all documents:

scores = X.dot(v_query)

# This is matrix-vector multiplication. Each element `i` of `scores` is the cosine similarity between document `i` (row `i` of `X`) and `v_query`

# We could compute the same thing with a for loop => [v_query.dot(X[i]) for i in range(len(X))]

# But X.dot(v_query) is much faster.

scores

array([ 0.48740596,  0.20991942,  0.7629412 , ..., -0.08637968,
        0.03759792, -0.03037032], shape=(1368,), dtype=float32)

### Best Match
---

The highest score is the most similar document

In [36]:
idx = np.argmax(scores)
idx, scores[idx]

# This returns document 553 with score 0.76.

# The index and score may differ for you, because this FAQ is living document, so we add and remove entries over time.



(np.int64(2), np.float32(0.7629412))

In [37]:
# We can use the index to get the document from the original dataset:

documents[2]



{'id': '3f1424af17',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

### Top 5 results
---

Usually we want more than the single best match, so let's pull the top 5.

`np.argsort` sorts from lowest to highest, so the last 5 are the top ones

In [38]:
top5 = np.argsort(scores)[-5:]

top5

array([  7, 538, 925, 643,   2])

In [39]:
# They come out smallest-first, so we reverse them to get the highest first:

top5 = top5[::-1]
top5

array([  2, 643, 925, 538,   7])

In [40]:
# Let's read off the actual documents:

for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.7629412
{'id': '3f1424af17', 'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

0.7579372
{'id': '2d8b16c2a0', 'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute."}

0.7192135
{'id': '41aabbd7c5', 'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related

This is vector search in its simplest form. We embed the query, compute dot products against all documents, and return the highest-scoring ones.

We return 5 and not the single best for a reason. The answer to a question can be spread across several documents. One holds part of it, another fills in the rest. Sometimes the top result isn't the right one but the second is. We send all 5 to the LLM and let it combine them.

The number 5 is a starting point, picked on gut feeling. Later, when we evaluate search quality, we can test whether 3 or 10 works better for our data.

Doing this by hand with numpy is fine for a small dataset. A larger one needs a library that also handles filtering and ranking. That